In [11]:
import pandas as pd

In [12]:
train_df = pd.read_csv("titanic/train.csv")
train_df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [13]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


In [14]:
train_df.isnull().sum()

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [15]:
def preprocess_data(df):    
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df.drop(['Cabin', 'Ticket', 'Name', 'PassengerId'], axis=1, inplace=True)
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})
    df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
    df['Embarked'] = df['Embarked'].map({'S': 0, 'C': 1, 'Q': 2})
    return df

In [16]:
train_df = preprocess_data(train_df)
train_df.head()

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,0,22.0,1,0,7.2500,0
1,1,1,1,38.0,1,0,71.2833,1
2,1,3,1,26.0,0,0,7.9250,0
3,1,1,1,35.0,1,0,53.1000,0
4,0,3,0,35.0,0,0,8.0500,0


### Random Forest with CV

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score

X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

model = RandomForestClassifier(n_estimators=100, max_depth=10, criterion='gini', random_state=42)

cv_results = cross_validate(model, X, y, cv=10, scoring='accuracy', return_estimator=True)

scores = cv_results['test_score']
models = cv_results['estimator']


print("All scores:", scores)
print(f"Best score: {scores.max():.4f} from fold {scores.argmax()}")

# Extract the best model
best_fold_index = scores.argmax()
best_model = models[best_fold_index]

print(f"\nBest model is from fold #{best_fold_index + 1}")
print(f"Best model accuracy: {scores[best_fold_index]:.4f}")

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# model.fit(X_train, y_train)
preds = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))

All scores: [0.76666667 0.82022472 0.74157303 0.86516854 0.93258427 0.83146067
 0.83146067 0.79775281 0.86516854 0.84269663]
Best score: 0.9326 from fold 4

Best model is from fold #5
Best model accuracy: 0.9326
Accuracy: 0.9441340782122905


### Grid search

In [18]:
# from sklearn.model_selection import GridSearchCV

# param_grid = {
#     'n_estimators': [100, 200, 500],
#     'max_depth': [None, 5, 10, 20],
#     'min_samples_split': [2, 5, 10],
#     'min_samples_leaf': [1, 2, 4],
#     'max_features': ['sqrt', 'log2'],
#     'bootstrap': [True, False]
# }

# rf = RandomForestClassifier(random_state=42)
# grid_search = GridSearchCV(rf, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
# grid_search.fit(X_train, y_train)

# model = grid_search.best_estimator_
# print("Best params:", grid_search.best_params_)
# preds = model.predict(X_val)
# print("Accuracy:", accuracy_score(y_val, preds))

### Grid Search with StratifiedKfold CV

In [19]:
# from sklearn.model_selection import StratifiedKFold, GridSearchCV
# from sklearn.metrics import classification_report
# import numpy as np

# cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# param_grid = {
#     'n_estimators': [80, 100, 200],
#     'max_depth': [5, 10, None],
#     'max_features': ['sqrt', 'log2', None],
#     'min_samples_split': [2, 5],
#     'min_samples_leaf': [1, 2]
# }

# base_model = RandomForestClassifier(criterion='gini', random_state=42, verbose=0)

# grid_search = GridSearchCV(
#     estimator=base_model,
#     param_grid=param_grid,
#     cv=cv,
#     scoring='accuracy',
#     n_jobs=-1,  # Use all CPU cores
#     verbose=1
# )

# grid_search.fit(X,y)

# print(f"Best parameters: {grid_search.best_params_}")
# print(f"Best cross-validation score: {grid_search.best_score_:.4f}")

# # Get detailed CV results
# cv_results = grid_search.cv_results_
# print(f"\nTop 3 models:")
# for i in np.argsort(cv_results['rank_test_score'])[:3]:
#     print(f"Rank {cv_results['rank_test_score'][i]}: "
#           f"Score={cv_results['mean_test_score'][i]:.4f} "
#           f"Params={cv_results['params'][i]}")

# best_model = grid_search.best_estimator_
# best_model.fit(X_train, y_train)
# y_pred = best_model.predict(X_val)

# print(f"\nHoldout test accuracy: {accuracy_score(y_val, y_pred):.4f}")
# print("\nClassification Report:")
# print(classification_report(y_val, y_pred))

### Adaboost

In [ ]:
# from sklearn.ensemble import AdaBoostClassifier
# from sklearn.tree import DecisionTreeClassifier

# ada_default = AdaBoostClassifier(
#     n_estimators=100,
#     learning_rate=0.1,
#     random_state=42
# )

# ada

NameError: name 'ada' is not defined

### XGBoost

In [21]:
# !pip install xgboost

Defaulting to user installation because normal site-packages is not writeable
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 5.0 MB/s eta 0:00:0000:0100:01


In [50]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=5,
    random_state=42
)

cv_results = cross_validate(xgb, X, y, cv=5, scoring='accuracy', return_estimator=True)

scores = cv_results['test_score']
models = cv_results['estimator']


print("All scores:", scores)
print(f"Best score: {scores.max():.4f} from fold {scores.argmax()}")

# Extract the best model
best_fold_index = scores.argmax()
best_model = models[best_fold_index]

print(f"\nBest model is from fold #{best_fold_index + 1}")
print(f"Best model accuracy: {scores[best_fold_index]:.4f}")

# xgb.fit(X_train, y_train)

preds = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))

All scores: [0.81005587 0.82022472 0.87078652 0.80898876 0.85955056]
Best score: 0.8708 from fold 2

Best model is from fold #3
Best model accuracy: 0.8708
Accuracy: 0.8938547486033519


### LightGBM

In [58]:
# !pip install lightgbm

In [83]:
from lightgbm import LGBMClassifier

lgbm = LGBMClassifier(
    n_estimators=100,
    learning_rate=.05,
    random_state=42,
    verbose=-1
)

cv_results = cross_validate(lgbm, X, y, cv=5, scoring='accuracy', return_estimator=True)

scores = cv_results['test_score']
models = cv_results['estimator']


print("All scores:", scores)
print(f"Best score: {scores.max():.4f} from fold {scores.argmax()}")

# Extract the best model
best_fold_index = scores.argmax()
best_model = models[best_fold_index]

print(f"\nBest model is from fold #{best_fold_index + 1}")
print(f"Best model accuracy: {scores[best_fold_index]:.4f}")


# lgbm.fit(X_train,y_train)
preds = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))

All scores: [0.81564246 0.8258427  0.87078652 0.83146067 0.85955056]
Best score: 0.8708 from fold 2

Best model is from fold #3
Best model accuracy: 0.8708
Accuracy: 0.9050279329608939


### CatBoost

In [89]:
# !pip install catboost

In [103]:
from catboost import CatBoostClassifier

catb = CatBoostClassifier(
    iterations=100,
    learning_rate=.1,
    max_depth=5,
    random_seed=42,
    verbose=False
)

cv_results = cross_validate(catb, X, y, cv=5, scoring='accuracy', return_estimator=True)

scores = cv_results['test_score']
models = cv_results['estimator']


print("All scores:", scores)
print(f"Best score: {scores.max():.4f} from fold {scores.argmax()}")

# Extract the best model
best_fold_index = scores.argmax()
best_model = models[best_fold_index]

print(f"\nBest model is from fold #{best_fold_index + 1}")
print(f"Best model accuracy: {scores[best_fold_index]:.4f}")

# catb.fit(X_train,y_train)
preds = best_model.predict(X_val)

print("Accuracy:", accuracy_score(y_val, preds))

All scores: [0.80446927 0.82022472 0.8258427  0.79775281 0.86516854]
Best score: 0.8652 from fold 4

Best model is from fold #5
Best model accuracy: 0.8652
Accuracy: 0.8212290502793296


### Predicting for test csv

In [52]:
test_df = pd.read_csv("titanic/test.csv")
test_df.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [53]:
test_df = preprocess_data(test_df)
test_df.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,0,34.5,0,0,7.8292,2
1,3,1,47.0,1,0,7.0000,0
2,2,0,62.0,0,0,9.6875,2
3,3,0,27.0,0,0,8.6625,0
4,3,1,22.0,1,1,12.2875,0


In [84]:
result = best_model.predict(test_df)
result[:5]

array([0, 0, 0, 0, 0])

In [85]:
submission_df = pd.read_csv("titanic/gender_submission.csv")
submission_df.head()

,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1


In [86]:
output = pd.DataFrame({'PassengerId': submission_df.PassengerId, 'Survived': result})
output.to_csv('submission.csv', index=False)
print("Your submission was successfully saved!")

Your submission was successfully saved!
